# Simple Campus Menu Agent

This notebook creates a simple agent that can answer questions about today's campus menu.
It uses the `get_menu` tool from `getmenus.py` to look up data from the database.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from IPython.display import display, Markdown
from loguru import logger

# Import the get_menu tool and helper functions from getmenus.py
from getmenus import get_menu, get_today, setup_logging

setup_logging()
load_dotenv()
print("Environment loaded.")

In [ ]:
logger.info(f"Today's date (Seoul time): {get_today()}")

## Create the Language Model

Connect to Azure OpenAI using credentials stored in `.env`.

In [ ]:
# Create the language model using Azure OpenAI credentials from the .env file
llm = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    timeout=300,
    temperature=0.5,
    max_tokens=25000
)

logger.info("LLM ready.")


## Build the Agent

Create the agent with the `get_menu` tool and a helpful system prompt.

In [ ]:
today = get_today()

# The system prompt tells the agent how to behave and what today's date is
SYSTEM_PROMPT = (
    f"You are a helpful campus menu assistant. Today's date is {today} (Asia/Seoul). "
    "Use the get_menu tool to fetch menu data for the requested date. "
    "Translate all Korean names (universities, restaurants, dishes) to English. "
    "Group the answer by university, then by restaurant. "
    "Include meal type, dish names, serving time, and price when available."
)

# Build the agent with the get_menu tool attached
agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_menu]
)

logger.info("Agent ready.")

## Ask the Agent a Question

Change the `question` variable below and run the cell to get an answer.

In [ ]:
# Change this question to ask anything about the menu
question = "What is on the menu today? Format the answer in HTML so that I can send it via email."

In [ ]:
logger.info("Question: {!r}", question)

# Run the agent with our question
result = agent.invoke({"messages": [{"role": "user", "content": question}]})

# Extract the final text reply from the agent's response messages
answer = ""
for message in reversed(result.get("messages", [])):
    content = getattr(message, "content", None)
    if isinstance(content, str) and content.strip():
        answer = content
        break

logger.info("Answer: {} chars", len(answer))

# Display the answer nicely formatted in the notebook
display(Markdown(answer))

In [ ]:
import httpx

def send_html_via_smtp2go(html_body: str, subject: str) -> dict:
    # Log function start so the step appears in logs/
    logger.info("Starting SMTP2GO email send function.")

    # Reload .env values (safe in notebooks) in case variables changed
    load_dotenv()

    # Read SMTP2GO configuration from environment variables
    smtp2go_api_key = os.getenv("SMTP2GO_API_KEY")
    smtp2go_sender_email = os.getenv("SMTP2GO_SENDER_EMAIL")
    smtp2go_recipient_email = os.getenv("SMTP2GO_RECIPIENT_EMAIL")

    # Validate required configuration before sending
    if not smtp2go_api_key:
        logger.error("Missing SMTP2GO_API_KEY in .env")
        raise ValueError("SMTP2GO_API_KEY is missing in .env")
    if not smtp2go_sender_email:
        logger.error("Missing SMTP2GO_SENDER_EMAIL in .env")
        raise ValueError("SMTP2GO_SENDER_EMAIL is missing in .env")
    if not smtp2go_recipient_email:
        logger.error("Missing SMTP2GO_RECIPIENT_EMAIL in .env")
        raise ValueError("SMTP2GO_RECIPIENT_EMAIL is missing in .env")

    logger.info("SMTP2GO configuration loaded successfully.")
    # Masked key for diagnostics (don't log full key)
    try:
        masked = smtp2go_api_key[:4] + '...' + smtp2go_api_key[-4:] if smtp2go_api_key else '****'
    except Exception:
        masked = '****'
    logger.info("SMTP2GO API key (masked): {}", masked)

    # SMTP2GO API endpoint and required headers
    endpoint_url = "https://api.smtp2go.com/v3/email/send"
    headers = {
        "Content-Type": "application/json",
        "accept": "application/json",
        "X-Smtp2go-Api-Key": smtp2go_api_key
    }

    # Build the email payload using html_body (include api_key for compatibility)
    payload = {
        "api_key": smtp2go_api_key,
        "sender": smtp2go_sender_email,
        "to": [smtp2go_recipient_email],
        "subject": subject,
        "html_body": html_body
    }

    logger.info(
        "Sending email via SMTP2GO. sender='{}' recipient='{}' subject='{}'",
        smtp2go_sender_email,
        smtp2go_recipient_email,
        subject
    )

    try:
        # Send the POST request to SMTP2GO
        response = httpx.post(endpoint_url, headers=headers, json=payload, timeout=30.0)
        logger.info("SMTP2GO response status code: {}", response.status_code)

        # Raise an exception for 4xx/5xx responses
        response.raise_for_status()

        # Parse JSON response and log result summary
        response_data = response.json()
        logger.info("SMTP2GO response parsed successfully.")
        logger.info("SMTP2GO request_id: {}", response_data.get("request_id"))

        data_block = response_data.get("data", {})
        logger.info(
            "SMTP2GO send result: succeeded={} failed={}",
            data_block.get("succeeded"),
            data_block.get("failed")
        )

        return response_data

    except httpx.HTTPStatusError as error:
        # Log detailed API error body if present
        logger.exception("SMTP2GO returned an HTTP error: {}", str(error))
        try:
            logger.error("SMTP2GO error response body: {}", error.response.text)
        except Exception:
            logger.error("SMTP2GO error response body: <unavailable>")
        raise
    except Exception as error:
        # Log unexpected errors
        logger.exception("Unexpected error while sending SMTP2GO email: {}", str(error))
        raise

In [ ]:
send_html_via_smtp2go(answer, subject="Today's Campus Menu")